# Complex CaDSD ABC: omega/rho search

This notebook is the clean version of the real experiment.

The design is changed by changing `omega` and `rho`, then building a new kernel with `CaDsd(omega, rho, pi)`. The objective is to improve the HT variance for `z`. We also report what happens to `y`.

## 1. Load the terminal runner

The long simulation code is kept in a `.py` file so it can run safely from the terminal and save resumable state files.

In [4]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "jupyters" else Path.cwd()
SCRIPT_PATH = PROJECT_ROOT / "simulations_abc" / "terminal_running" / "run_cadsd_complex_abc_vincent.py"

spec = importlib.util.spec_from_file_location("complex_abc", SCRIPT_PATH)
complex_abc = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = complex_abc
spec.loader.exec_module(complex_abc)

print(SCRIPT_PATH)

/home/bardia/projects/graphical-sampling/simulations_abc/terminal_running/run_cadsd_complex_abc_vincent.py


## 2. Population and cases

The current default uses twelve simulated cases: four `z` variables (`z_00`, `z_80`, `z_90`, `z_99`) and three inclusion-probability sources (`equal`, `size_80`, `size_90`).

In [6]:
df_pop, z_names, pi_sources = complex_abc.make_simulated_population()

print("z variables:", z_names)
print("pi sources:", pi_sources)

df_pop

z variables: ['z_00', 'z_80', 'z_90']
pi sources: ['equal', 'size_80', 'size_90']


,unit,y,z_00,z_80,size_80,z_90,size_90
0,0,80.120250,-0.520740,-0.655509,0.226526,-0.786855,0.176260
1,1,88.510461,-1.403829,0.356914,-0.717030,0.379050,0.195186
2,2,105.583754,0.530832,0.703437,1.283464,1.666042,2.255536
3,3,78.768159,-1.087365,-0.022793,-0.651279,-0.249948,-0.714172
4,4,71.667155,0.882960,-0.494720,-0.415759,-0.951502,-0.903462
5,5,123.206308,-0.840016,2.390054,2.733490,1.944190,2.247679
6,6,87.348379,0.355529,0.168885,0.990863,0.159678,0.581821
7,7,92.875923,-0.427927,1.382832,0.417302,-0.080592,-0.212394
8,8,75.293252,1.168233,0.014189,-1.220072,-1.113908,-0.644629
9,9,82.301236,0.562944,-0.780350,-1.451213,0.257695,-0.090910


## 3. What one ABC step means

For each candidate design:

1. ABC mutates `omega` and `rho`.
2. `CaDsd(omega, rho, pi)` creates a possibly complex kernel `K`.
3. We compute the HT variance for `z` and `y`.
4. ABC keeps candidates that improve the objective for `z`.

The starting point is `omega = 0`, `rho = 0.5`, which should match the Vincent/Ppi benchmark for `z`.

## 4. Run one small case inside the notebook

This is only for checking. Long runs should use the terminal command in the next section.

In [8]:
case_args = (
    "notebook_demo",   # state/output stem
    "z_90",            # z variable
    "size_90",         # pi source
    300,                  # target iterations
    1,                  # checkpoint interval
    4,                  # colony size
    5,                  # abandonment limit
    0.5,                # onlooker factor
    10,                 # local search interval
    1,                  # local search attempts
    "fast",            # validation mode
    20260706,           # seed
    False,              # resume
)

final_row, checkpoint_rows, current_iteration = complex_abc.run_case(case_args)
pd.DataFrame([final_row])

,iteration,z_variable,pi_source,corr_y_z,corr_y_pi,corr_y_over_pi_z_over_pi,Start_z_over_Vincent_z,Start_y_over_Vincent_y,ABC_z_over_Vincent_z,ABC_y_over_Vincent_y,...,ABC_eff_z,ABC_eff_y,Random_eff_z,Random_eff_y,Vincent_eff_z,Vincent_y_eff_y,evaluations_ABC,evaluations_Random,valid_ABC,valid_Random
0,300,z_90,size_90,0.9,0.871442,-0.542058,1.0,0.600761,1.145777,0.245405,...,1.907133,1.496222,1.664487,3.662802,1.664488,6.096942,5484,5484,5484,5478


## 5. Run the full experiment inside this notebook

Run this cell to execute the selected cases directly in Jupyter. It uses the same omega/rho ABC code as the terminal script and saves both final results and checkpoints.

In [10]:
# Settings you can change
ITERATIONS = 500
CHECKPOINT_INTERVAL = 20
COLONY_SIZE = 8
LIMIT = 5
ONLOOKER_FACTOR = 0.5
LOCAL_SEARCH_INTERVAL = 10
LOCAL_SEARCH_ATTEMPTS = 1
VALIDATION_MODE = "fast"
BASE_SEED = 20260706
RESUME = False  # change to True to continue from saved state

# Choose cases. Use z_names/pi_sources for all cases, or write lists like ["z_90"].
SELECTED_Z = z_names
SELECTED_PI = pi_sources

stem = f"complex_cadsd_notebook_{COLONY_SIZE}col_{VALIDATION_MODE}"
artifact_dir = PROJECT_ROOT / "simulations_abc" / "jupyters" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

final_path = artifact_dir / f"{stem}_{ITERATIONS}iter.csv"
checkpoint_path = artifact_dir / f"{stem}_{ITERATIONS}iter_checkpoints.csv"

cases = []
for z_i, z_var in enumerate(SELECTED_Z):
    for p_i, pi_source in enumerate(SELECTED_PI):
        cases.append((
            stem,
            z_var,
            pi_source,
            ITERATIONS,
            CHECKPOINT_INTERVAL,
            COLONY_SIZE,
            LIMIT,
            ONLOOKER_FACTOR,
            LOCAL_SEARCH_INTERVAL,
            LOCAL_SEARCH_ATTEMPTS,
            VALIDATION_MODE,
            BASE_SEED + 1000 * z_i + 17 * p_i,
            RESUME,
        ))

finals = []
all_checkpoints = []

for i, case_args in enumerate(cases, start=1):
    final_row, checkpoint_rows, current_iteration = complex_abc.run_case(case_args)
    finals.append(final_row)
    all_checkpoints.extend(checkpoint_rows)
    print(
        f"{i}/{len(cases)} {final_row['z_variable']} / {final_row['pi_source']} "
        f"iter={current_iteration}: "
        f"ABC_z/V={final_row['ABC_z_over_Vincent_z']:.4f}, "
        f"ABC_y/Vy={final_row['ABC_y_over_Vincent_y']:.4f}, "
        f"Random_z/V={final_row['Random_z_over_Vincent_z']:.4f}"
    )

final_df = pd.DataFrame(finals).sort_values(["z_variable", "pi_source"]).reset_index(drop=True)
checkpoint_df = pd.DataFrame(all_checkpoints).sort_values(["iteration", "z_variable", "pi_source"]).reset_index(drop=True)

final_df.to_csv(final_path, index=False)
checkpoint_df.to_csv(checkpoint_path, index=False)

print("Saved final results to:", final_path)
print("Saved checkpoints to:", checkpoint_path)

display(final_df)

KeyboardInterrupt: 

To continue later in the notebook:

1. Increase `ITERATIONS`, for example from `50` to `100`.
2. Set `RESUME = True`.
3. Run the full experiment cell again.

The state files are saved under `simulations_abc/jupyters/artifacts/complex_cadsd_states/`.

## 6. Read saved results

In [ ]:
result_path = PROJECT_ROOT / "simulations_abc" / "jupyters" / "artifacts" / "complex_cadsd_notebook_8col_fast_50iter.csv"
if result_path.exists():
    display(pd.read_csv(result_path))
else:
    print("Run the full experiment cell first.")